In [20]:
### ============ 1. IMPORTS =====================
import pandas as pd # for data manipulation
import pickle # for saving the processed data to disk
import sys # for manipulating the Python path
import os # for file path manipulations
from scipy.io import arff # for loading ARFF files from OpenML

# Make the src/ package importable from within the notebooks/ subfolder.
sys.path.append("..")

# Pull in the same SEED and TEST_SIZE constants from config.py so that all notebooks use the same values.
from src.config import SEED, TEST_SIZE
from src.preprocess import split_data, scale_data

In [ ]:
#  ===============DATASET CONFIGURATION=====================
# DATASET_NAME = "dataset1"
# FILE_PATH = f"../data/raw/{DATASET_NAME}.arff"
# TARGET_COL = "CLASS_LABEL" 

### 3.LOAD DATA

In [ ]:
# ===================== DATASET 1 - UCI Phishing Website Dataset (Full Variant) [16] =====
# Load the data file
# data, meta = arff.loadarff('../data/raw/Phishing_Legitimate_full.arff')
# # Convert to a Pandas data frame
# df = pd.DataFrame(data)
# # inspect the data
# df.head()

,NumDots,SubdomainLevel,PathLevel,UrlLength,NumDash,NumDashInHostname,AtSymbol,TildeSymbol,NumUnderscore,NumPercent,...,IframeOrFrame,MissingTitle,ImagesOnlyInForm,SubdomainLevelRT,UrlLengthRT,PctExtResourceUrlsRT,AbnormalExtFormActionR,ExtMetaScriptLinkRT,PctExtNullSelfRedirectHyperlinksRT,CLASS_LABEL
0,3.0,1.0,5.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,1.0,-1.0,1.0,b'1'
1,3.0,1.0,3.0,144.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,1.0,-1.0,1.0,1.0,1.0,1.0,b'1'
2,3.0,1.0,2.0,58.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,-1.0,1.0,-1.0,0.0,b'1'
3,3.0,1.0,6.0,79.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,-1.0,1.0,1.0,1.0,-1.0,b'1'
4,3.0,0.0,4.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,-1.0,0.0,-1.0,-1.0,b'1'


In [ ]:
# DATASET 3 - UCI Phishing Website Dataset (Small Variant) [18]
# # load the data file
# df = pd.read_csv('../data/raw/dataset_small.csv')
# # inspect the data
# print(df.shape)
# df.head()

(58645, 112)


,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,qty_exclamation_url,qty_space_url,...,qty_ip_resolved,qty_nameservers,qty_mx_servers,ttl_hostname,tls_ssl_certificate,qty_redirects,url_google_index,domain_google_index,url_shortened,phishing
0,2,0,0,0,0,0,0,0,0,0,...,1,4,2,3598,0,0,0,0,0,0
1,4,0,0,2,0,0,0,0,0,0,...,1,4,1,3977,1,0,0,0,0,0
2,1,0,0,1,0,0,0,0,0,0,...,1,2,1,10788,0,0,0,0,0,0
3,2,0,0,3,0,0,0,0,0,0,...,1,2,1,14339,1,0,0,0,0,1
4,1,1,0,4,0,0,0,0,0,0,...,1,2,1,389,1,1,0,0,0,1


In [21]:
# DATASET 4 - UCI Phishing Website Dataset (Full Variant) [19]
# load the data file
df = pd.read_csv('../data/raw/dataset_full.csv')
# inspect the data
print(df.shape)
df.head()

(88647, 112)


,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,qty_exclamation_url,qty_space_url,...,qty_ip_resolved,qty_nameservers,qty_mx_servers,ttl_hostname,tls_ssl_certificate,qty_redirects,url_google_index,domain_google_index,url_shortened,phishing
0,3,0,0,1,0,0,0,0,0,0,...,1,2,0,892,0,0,0,0,0,1
1,5,0,1,3,0,3,0,2,0,0,...,1,2,1,9540,1,0,0,0,0,1
2,2,0,0,1,0,0,0,0,0,0,...,1,2,3,589,1,0,0,0,0,0
3,4,0,2,5,0,0,0,0,0,0,...,1,2,0,292,1,0,0,0,0,1
4,2,0,0,0,0,0,0,0,0,0,...,1,2,1,3597,0,1,0,0,0,0


In [22]:
# ===== 3. DATA INSPECTION =====
# check the shape to confirm we have the expected number of rows and columns
print(df.shape)
# show summary statistics and data types to check for any obvious issues
print(df.info()) 
# check for missing values and duplicates
print(df.isnull().sum())
#  check if there are duplicates so they can be removed before training the GA
print(df.duplicated().sum())

(88647, 112)
<class 'pandas.DataFrame'>
RangeIndex: 88647 entries, 0 to 88646
Columns: 112 entries, qty_dot_url to phishing
dtypes: float64(1), int64(111)
memory usage: 75.7 MB
None
qty_dot_url             0
qty_hyphen_url          0
qty_underline_url       0
qty_slash_url           0
qty_questionmark_url    0
                       ..
qty_redirects           0
url_google_index        0
domain_google_index     0
url_shortened           0
phishing                0
Length: 112, dtype: int64
1438


In [23]:
# ===== 4. PREPROCESSING =====
### Identify target and features
target_col = "phishing"

# Drop the label column to get the feature matrix X.
X = df.drop(columns=[target_col])

# convert bytes to int
y = df[target_col].astype(str).astype(int)

print("X shape:", X.shape[1])   # number of features
print("y shape:", y.shape)      # number of samples

# check the distribution of the target variable
print(y.value_counts())
print(y.unique())

X shape: 111
y shape: (88647,)
phishing
0    58000
1    30647
Name: count, dtype: int64
[1 0]


In [ ]:
# DATASET 2 - UCI Phishing Website Dataset [17]
# # load the data file
# data, meta = arff.loadarff('../data/raw/.old.arff')

# # Convert to a Pandas data frame
# df = pd.DataFrame(data)

# # convert bytes to int
# for col in df.select_dtypes(include=["object"]).columns:
#     df[col] = df[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

# # convert to int
# df = df.astype(int)

# # fix the target variable to be binary 0/1 instead of -1/1
# df["Result"] = df["Result"].replace({-1: 0, 1: 1})

# # inspect the data
# df.head()

# # split the data into features and target
# X = df.drop(columns=["Result"])
# y = df["Result"]

In [24]:
# ===== 6. TRAIN / TEST SPLIT =====
# Use the same SEED and TEST_SIZE constants from config.py to ensure all notebooks use the same split.
X_train, X_test, y_train, y_test = split_data(X, y, test_size=TEST_SIZE, seed=SEED)
print(X_train.shape, X_test.shape)  # confirm 70/30 ratio looks right

(62052, 111) (26595, 111)


In [25]:
# ===== 7. SCALE FEATURES =====
# scale the data
X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

# Verify the scaling worked by checking the min and max values of the scaled features.
print(X_train_scaled.describe().loc[["min", "max"]])

     qty_dot_url  qty_hyphen_url  qty_underline_url  qty_slash_url  \
min          0.0             0.0                0.0            0.0   
max          1.0             1.0                1.0            1.0   

     qty_questionmark_url  qty_equal_url  qty_at_url  qty_and_url  \
min                   0.0            0.0         0.0          0.0   
max                   1.0            1.0         1.0          1.0   

     qty_exclamation_url  qty_space_url  ...  time_domain_expiration  \
min                  0.0            0.0  ...                     0.0   
max                  1.0            1.0  ...                     1.0   

     qty_ip_resolved  qty_nameservers  qty_mx_servers  ttl_hostname  \
min              0.0              0.0             0.0           0.0   
max              1.0              1.0             1.0           1.0   

     tls_ssl_certificate  qty_redirects  url_google_index  \
min                  0.0            0.0               0.0   
max                  1.0    

In [26]:
# ===== 8. SAVE PROCESSED DATA =====
# Persist the split and scaled arrays to disk with pickle so that
# subsequent notebooks (02, 03, 04) can reload them without re-running
# the entire preprocessing pipeline from scratch.
with open("../data/processed/uci_full_split_scaled.pkl", "wb") as f:
    pickle.dump((X_train_scaled, X_test_scaled, y_train, y_test), f)